In [1]:
import sys
sys.path.append('..') 

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from src.utils import set_seed, get_device, extract_features, train_clustering

set_seed(42)
device = get_device()
print(f"[INFO] Đang chạy trên thiết bị: {device}")

# Khởi tạo Backbone ResNet-18
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
backbone = nn.Sequential(*list(resnet.children())[:-1]).to(device)

# Tải dữ liệu CIFAR-10 và trích xuất vector Z
transform_cifar = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
cifar_data = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform_cifar)
cifar_subset = torch.utils.data.Subset(cifar_data, range(5000))
cifar_loader = DataLoader(cifar_subset, batch_size=128, shuffle=False)

print("[INFO] Đang trích xuất đặc trưng...")
z_cifar, y_cifar = extract_features(cifar_loader, backbone, device)
print("[INFO] Trích xuất xong! Sẵn sàng chạy Ablation.")

[INFO] Đang chạy trên thiết bị: cpu
[INFO] Đang trích xuất đặc trưng...
[INFO] Trích xuất xong! Sẵn sàng chạy Ablation.


In [2]:
print("\n--- THỰC NGHIỆM 2: ABLATION STUDY (TẮT CƠ CHẾ TÁCH/GỘP) ---")
# Thiết lập k_max = 10 và apply_sparsity = False
k_star_abl, acc_abl, nmi_abl, ari_abl = train_clustering(
    features=z_cifar, 
    labels=y_cifar, 
    k_max=10, 
    device=device, 
    apply_sparsity=False, 
    epochs=50
)

print(f"Gán cứng K=10 từ đầu | K* cuối cùng: {k_star_abl}")
print(f"Kết quả Ablation: ACC = {acc_abl*100:.2f}% | NMI = {nmi_abl*100:.2f}% | ARI = {ari_abl*100:.2f}%")


--- THỰC NGHIỆM 2: ABLATION STUDY (TẮT CƠ CHẾ TÁCH/GỘP) ---
Gán cứng K=10 từ đầu | K* cuối cùng: 3
Kết quả Ablation: ACC = 27.58% | NMI = 39.71% | ARI = 19.04%
